In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import *

In [0]:
# Leitura dos arquivos Parquet na pasta Novos
df = spark.read.parquet(f"/Volumes/workspace/landing/currency_analysis/Cotacoes/Novos/*.parquet")

df.createOrReplaceTempView("df")

df = spark.sql("""
    SELECT 
        CAST(dataHoraCotacao AS DATE) AS dt_periodo,
        moeda AS dc_moeda,
        cotacaoCompra AS vl_cotacao
    FROM
        df
    ORDER BY dt_periodo ASC
""").dropDuplicates(["dc_moeda", "dt_periodo"])

display(df)

In [0]:
%sql
-- Cria uma tabela no lake se não existir
CREATE TABLE IF NOT EXISTS `02_silver_currency`.cotacoes (
    dt_periodo DATE,
    dc_moeda STRING,
    vl_cotacao DOUBLE
)
USING DELTA
PARTITIONED BY (dc_moeda)

In [0]:
# Executa o merge na tabela cotacoes
df.createOrReplaceTempView("df_novos")

spark.sql("""
    MERGE INTO `02_silver_currency`.cotacoes AS cotacoes
    USING (
        SELECT
            dt_periodo,
            dc_moeda,
            vl_cotacao
        FROM
            df_novos
    ) as novos
    ON cotacoes.dc_moeda = novos.dc_moeda
        AND cotacoes.dt_periodo = novos.dt_periodo
    WHEN NOT MATCHED THEN
        INSERT (dt_periodo, dc_moeda, vl_cotacao)
        VALUES (novos.dt_periodo,novos.dc_moeda,novos.vl_cotacao)
""")

In [0]:
origem = "/Volumes/workspace/landing/currency_analysis/Cotacoes/Novos/"
destino = "/Volumes/workspace/landing/currency_analysis/Cotacoes/Carregados/"

# cria a pasta destino se não existir
try:
    dbutils.fs.ls(destino)
except Exception:
    dbutils.fs.mkdirs(destino)

# move recursivamente todo o conteúdo
dbutils.fs.mv(origem, destino, recurse=True)
